#### HF Losses in Ferrite Core Cross sections


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mph
import mphsweepkit as msk
import numpy as np

Start the Comsol Server and load the model

In [3]:
# Start the COMSOL client
client = mph.start()

In [4]:
# Load the model
model = client.load('core_cross_section_initial.mph')

Create a CascadedSweepModel

In [5]:
# Create a CascadedSweepModel object
csm = msk.CascadedSweepModel(model, 'Study on Cross-Sections', show_param_names=True)

Initialized CascadedSweepModel
Study name: Study on Cross-Sections
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'hor_slit', 'vert_slit', 'w', 'l_r', 'a_e'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.core'
        - Excitation Sweep (Parametric) -> 'b_mean'
          - Frequency Sweep (Frequency)
--------------------------------
Data updated from MPh-model.
Input data shape: (312, 8)
Reset output data to shape of the input data: (312, 0)
Combined shape: (312, 8)


Change the material to 3F46

In [6]:
csm.set_material_sweep(sweep_name='Material Sweep', 
                       material_names=['matsw.comp1.core'], 
                       material_values=[[2]], 
                       sweep_type="filled")

csm.input_data

--------------------------------
Data updated from MPh-model.
Input data shape: (156, 8)
Reset output data to shape of the input data: (156, 0)
Combined shape: (156, 8)


unit,hor_slit,vert_slit,w,l_r,a_e,matsw.comp1.core,b_mean,freq
name,um,um,mm,mm,mm,,mT,kHz
group,Geometry Sweep,Geometry Sweep,Geometry Sweep,Geometry Sweep,Geometry Sweep,Material Sweep,Excitation Sweep,Frequency Sweep
0,0.0,0.0,5.0,0.0,5.0,2.0,25.0,100.0
1,0.0,0.0,5.0,0.0,5.0,2.0,25.0,200.0
2,0.0,0.0,5.0,0.0,5.0,2.0,25.0,300.0
3,0.0,0.0,5.0,0.0,5.0,2.0,25.0,400.0
4,0.0,0.0,5.0,0.0,5.0,2.0,25.0,500.0
...,...,...,...,...,...,...,...,...
151,0.0,0.0,15.0,0.0,15.0,2.0,100.0,900.0
152,0.0,0.0,15.0,0.0,15.0,2.0,100.0,1000.0


Set geometric sweep parameters to get circles with radii: 5, 10, 15 mm

In [7]:
# Definition of the radii as a helper:
param__radii = np.array([5, 10, 15])  # Radii for the cross-section in mm

# Create arrays for the parameters based on the radii
param__hor_slit = np.zeros_like(param__radii)  # horizontal slit 
param__vert_slit = np.zeros_like(param__radii)  # vertical slit 
param__w = param__radii * 2  # width 
param__l_r = np.zeros_like(param__radii)  # length 
param__a_e = param__radii  # large semi-axis of ellipse

# Set the parametric sweep for the 'Geometry Sweep'
csm.set_parametric_sweep(sweep_name='Geometry Sweep',
                         param_names=["hor_slit", "vert_slit", "w", "l_r", "a_e"], 
                         param_units=['um', 'um', 'mm', 'mm', 'mm'],
                         param_values=[param__hor_slit, 
                                       param__vert_slit, 
                                       param__w, 
                                       param__l_r, 
                                       param__a_e],
                         sweep_type="sparse")

csm.input_data


--------------------------------
Data updated from MPh-model.
Input data shape: (156, 8)
Reset output data to shape of the input data: (156, 0)
Combined shape: (156, 8)


unit,hor_slit,vert_slit,w,l_r,a_e,matsw.comp1.core,b_mean,freq
name,um,um,mm,mm,mm,,mT,kHz
group,Geometry Sweep,Geometry Sweep,Geometry Sweep,Geometry Sweep,Geometry Sweep,Material Sweep,Excitation Sweep,Frequency Sweep
0,0.0,0.0,10.0,0.0,5.0,2.0,25.0,100.0
1,0.0,0.0,10.0,0.0,5.0,2.0,25.0,200.0
2,0.0,0.0,10.0,0.0,5.0,2.0,25.0,300.0
3,0.0,0.0,10.0,0.0,5.0,2.0,25.0,400.0
4,0.0,0.0,10.0,0.0,5.0,2.0,25.0,500.0
...,...,...,...,...,...,...,...,...
151,0.0,0.0,30.0,0.0,15.0,2.0,100.0,900.0
152,0.0,0.0,30.0,0.0,15.0,2.0,100.0,1000.0


Simulation

In [32]:
csm.simulate()

Directory 'batch_data' already exists. Using existing directory.
Batch directory for the geometric sweep set to: x:\Till_data\repositories\MPhSweepKit\examples\studies\infinite_ferrite_cross_sections\batch_data


Save the model with solution data:

In [35]:
model.save('core_cross_section_solved.mph')

Release the Client

In [36]:
client.remove(model)
client.clear()
client.disconnect()